# 02 - Image Classification (Week 2)
Training a MobileNetV3 classifier on periapical lesion images (PAI 3, 4, 5).


In [ ]:
import sys, os, json
# Find project root (works from project root or notebooks/ subdir)
_root = None
for _p in [os.path.abspath('.'), os.path.abspath('..')]:
    if os.path.exists(os.path.join(_p, 'src', 'config.py')):
        _root = _p
        break
if _root:
    sys.path.append(_root)
else:
    raise RuntimeError('Could not find project root (src/config.py not found)')
from src.config import *
from src.train_classification import train_classification, evaluate_classification
from src.visualize import plot_training_history, plot_confusion_matrix, visualize_sample_predictions
import warnings; warnings.filterwarnings('ignore')


In [ ]:
# Load classification manifest
manifest_path = os.path.join(DATA_PROCESSED, 'classification', 'manifest.json')
with open(manifest_path) as f:
    all_records = json.load(f)
print(f'Loaded {len(all_records)} classification records')


In [ ]:
# Split records by split field
train_records = [r for r in all_records if r['split'] == 'train']
val_records = [r for r in all_records if r['split'] == 'val']
test_records = [r for r in all_records if r['split'] == 'test']
print(f'Train: {len(train_records)}, Val: {len(val_records)}, Test: {len(test_records)}')


In [ ]:
# Train classifier
model, history = train_classification(train_records, val_records)


In [ ]:
# Plot training history
plot_training_history(history, 'Classification',
    os.path.join(OUTPUTS_DIR, 'classification', 'training_history.png'))
print('Training history plot saved.')


In [ ]:
# Evaluate on test set
metrics, preds, labels, probs = evaluate_classification(model, test_records)


In [ ]:
# Plot confusion matrix
plot_confusion_matrix(metrics['confusion_matrix'],
    [CLASS_NAMES[i] for i in [3,4,5]],
    os.path.join(OUTPUTS_DIR, 'classification', 'confusion_matrix.png'))
print('Confusion matrix saved.')


In [ ]:
# Visualize sample predictions
visualize_sample_predictions(model, test_records,
    output_dir=os.path.join(OUTPUTS_DIR, 'classification'))
print('Sample predictions saved.')


In [ ]:
# Save metrics as JSON for GUI
with open(os.path.join(OUTPUTS_DIR, 'classification', 'metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)
print('Classification complete!')
